<a href="https://colab.research.google.com/github/Rogerio-mack/IA_2026S1/blob/main/mlp_penguins_solucao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="http://meusite.mackenzie.br/rogerio/mackenzie_logo/UPM.2_horizontal_vermelho.jpg"  width=300, align="right">


# Lab Multi Layer Perceptron

Neste exercício você vai empregar redes MLP do `scikit-learn` para resolver um problema de classificação sobre a base de dados `penguins`. Siga o modelo fornecido.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Modelo MLP

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

## Preparação dos dados



In [ ]:
# Carregando os dados
df = pd.read_csv('https://vincentarelbundock.github.io/Rdatasets/csv/MASS/Cars93.csv',index_col=0)
df = df.reset_index(drop=True)

### Dropna

Ou algum outro tratamento para valores ausentes.

In [ ]:
df.isna().sum()

,0
Manufacturer,0
Model,0
Type,0
Min.Price,0
Price,0
Max.Price,0
MPG.city,0
MPG.highway,0
AirBags,34
DriveTrain,0


Não vamos empregar esses dados com valores ausentes, então, não é necessário excluir as linhas.

In [ ]:
# Separando preditores e classe objetivo
X = df[['MPG.city', 'Horsepower', 'Type', 'Price', 'Width', 'Length', 'Weight', 'EngineSize', 'DriveTrain']]
y = df['Origin']

display(pd.concat([X,y],axis=1).head())

# Separando os dados de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=1)

,MPG.city,Horsepower,Type,Price,Width,Length,Weight,EngineSize,DriveTrain,Origin
0,25,140,Small,15.9,68,177,2705,1.8,Front,non-USA
1,18,200,Midsize,33.9,71,195,3560,3.2,Front,non-USA
2,20,172,Compact,29.1,67,180,3375,2.8,Front,non-USA
3,19,172,Midsize,37.7,70,193,3405,2.8,Front,non-USA
4,22,208,Midsize,30.0,69,186,3640,3.5,Rear,non-USA


### Hot encode dos dados

Assim como na normalização, fazer o `fit_transform()` nos dados de treinamento e a apenas o `transform()` nos dados de teste para evitar o *leak* de informações para o treinamento.

In [ ]:
# Aplicando one-hot encode
encoder = OneHotEncoder(sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train[['Type', 'DriveTrain']])
X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out())
X_train = pd.concat([X_train.drop(columns=['Type', 'DriveTrain']).reset_index(drop=True), X_train_encoded], axis=1)
display(X_train.head())

X_test_encoded = encoder.transform(X_test[['Type', 'DriveTrain']])
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out())
X_test = pd.concat([X_test.drop(columns=['Type', 'DriveTrain']).reset_index(drop=True), X_test_encoded], axis=1)
display(X_test.head())

,MPG.city,Horsepower,Price,Width,Length,Weight,EngineSize,Type_Compact,Type_Large,Type_Midsize,Type_Small,Type_Sporty,Type_Van,DriveTrain_4WD,DriveTrain_Front,DriveTrain_Rear
0,26,164,16.5,69,184,2970,2.5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,18,202,26.1,70,190,3730,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,19,170,24.4,74,177,3495,3.8,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,17,151,19.1,74,190,4100,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,33,73,8.4,60,146,2045,1.2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


,MPG.city,Horsepower,Price,Width,Length,Weight,EngineSize,Type_Compact,Type_Large,Type_Midsize,Type_Small,Type_Sporty,Type_Van,DriveTrain_4WD,DriveTrain_Front,DriveTrain_Rear
0,17,255,32.5,69,169,2895,1.3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,30,90,12.5,67,164,2475,1.6,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,29,110,11.8,66,170,2545,1.6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,19,200,18.5,72,195,3450,3.4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,23,110,11.1,66,181,2575,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Normalizando os dados

Vamos adotar a normalização *z-score* (StandardScaler), $x_{scaled} = \frac{x - \bar{x}}{\sigma(x)}$

In [ ]:
# Normalizando os dados
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
display(pd.DataFrame(X_train).head())
display(pd.DataFrame(X_test).head())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.575704,0.432470,-0.308597,-0.108099,0.107916,-0.186301,-0.173359,2.345208,-0.374634,-0.523937,-0.571429,-0.400892,-0.374634,-0.400892,0.642685,-0.426401
1,-0.731572,1.162114,0.661014,0.152139,0.504966,1.055704,0.300101,-0.426401,-0.374634,1.908627,-0.571429,-0.400892,-0.374634,-0.400892,0.642685,-0.426401
2,-0.568162,0.547677,0.489312,1.193089,-0.355309,0.671663,1.057637,-0.426401,2.669270,-0.523937,-0.571429,-0.400892,-0.374634,-0.400892,0.642685,-0.426401
3,-0.894981,0.182854,-0.045994,1.193089,0.504966,1.660364,0.300101,-0.426401,-0.374634,-0.523937,-0.571429,-0.400892,2.669270,-0.400892,0.642685,-0.426401
4,1.719570,-1.314837,-1.126707,-2.450236,-2.406733,-1.697951,-1.404356,-0.426401,-0.374634,-0.523937,1.750000,-0.400892,-0.374634,2.494438,-1.555973,-0.426401


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-0.894981,2.179776,1.307421,-0.108099,-0.884709,-0.308867,-1.309664,-0.426401,-0.374634,-0.523937,-0.571429,2.494438,-0.374634,-0.400892,-1.555973,2.345208
1,1.229342,-0.988417,-0.712602,-0.628574,-1.215583,-0.995238,-1.025588,-0.426401,-0.374634,-0.523937,-0.571429,2.494438,-0.374634,-0.400892,0.642685,-0.426401
2,1.065932,-0.604394,-0.783303,-0.888811,-0.818534,-0.880843,-1.025588,-0.426401,-0.374634,-0.523937,1.750000,-0.400892,-0.374634,-0.400892,0.642685,-0.426401
3,-0.568162,1.123712,-0.106595,0.672614,0.835841,0.598123,0.678869,-0.426401,-0.374634,1.908627,-0.571429,-0.400892,-0.374634,-0.400892,0.642685,-0.426401
4,0.085476,-0.604394,-0.854003,-0.888811,-0.090609,-0.831816,-0.646819,2.345208,-0.374634,-0.523937,-0.571429,-0.400892,-0.374634,-0.400892,0.642685,-0.426401


### Treinamento do Modelo

Definição do modelo e seus *Hiperparâmetros*, e o ajuste (treinamento) do Modelo.

In [ ]:
# Define o modelo de classificação (MLP)
clf = MLPClassifier(hidden_layer_sizes=(8,16,8), random_state=1, max_iter=10000)

# Treinando o modelo
clf.fit(X_train, y_train)

MLPClassifier(hidden_layer_sizes=(8, 16, 8), max_iter=10000, random_state=1)

### Métricas de Eficiência do Modelo

Acuracidade, Classification Report, sobre o conjunto de teste.


In [ ]:
# Calculando métricas de avaliação
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print("Acurácia:", accuracy)
print("Classification Report:\n", classification_rep)


Acurácia: 0.6071428571428571
Classification Report:
               precision    recall  f1-score   support

         USA       0.62      0.57      0.59        14
     non-USA       0.60      0.64      0.62        14

    accuracy                           0.61        28
   macro avg       0.61      0.61      0.61        28
weighted avg       0.61      0.61      0.61        28



### Predição

Faça a predição dos seguintes veículos em `X_new`:

In [ ]:
#@markdown Execute para gerar `X_new`
X_new = df[ df.Model.isin(['Mustang','Accord']) ][X.columns].reset_index(drop=True)



In [ ]:
display(X_new)

,MPG.city,Horsepower,Type,Price,Width,Length,Weight,EngineSize,DriveTrain
0,22,105,Sporty,15.9,68,180,2850,2.3,Rear
1,24,140,Compact,17.5,67,185,3040,2.2,Front


**Atenção**: Antes de aplicar a predição é necessário aplicar as mesmas transformações empregadas antes durante o treinamento.

In [ ]:
X_encoded = encoder.transform(X_new[['Type', 'DriveTrain']])
X_encoded = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out())
X_new = pd.concat([X_new.drop(columns=['Type', 'DriveTrain']), X_encoded], axis=1)
display(X_new.head())

X_new = scaler.transform(X_new)

y_pred = clf.predict(X_new)

print(y_pred)

,MPG.city,Horsepower,Price,Width,Length,Weight,EngineSize,Type_Compact,Type_Large,Type_Midsize,Type_Small,Type_Sporty,Type_Van,DriveTrain_4WD,DriveTrain_Front,DriveTrain_Rear
0,22,105,15.9,68,180,2850,2.3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,24,140,17.5,67,185,3040,2.2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


['USA' 'non-USA']


# Exercício

Empregue os modelos de rede neural MLP abaixo para classificar a origem, ilha do Pinguim, baseado em todos os demais atributos.


In [ ]:
df = sns.load_dataset('penguins')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## Preparação dos Dados

Exclua os dados ausentes se houverem, excluindo as linhas.


### Dropna

In [ ]:
df.isnull().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [ ]:
len_pre_drop = len(df)

In [ ]:
df = df.dropna().reset_index(drop=True)


In [ ]:
print(f'Número de linhas com valores ausentes: {len_pre_drop - len(df)}')

Número de linhas com valores ausentes: 11


In [ ]:
#@markdown Verificação, precisa retornar True
len(df) == 333

True

### Dados de treinamento e teste

Empregue 30% dos dados para teste, `random_state=1` e conjuntos estratificados pelo atributo classe (variável objetivo).

In [ ]:
# Separando os dados de treinamento e teste
X = df.drop(columns='island')
y = df['island']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=1)

In [ ]:
#@markdown Verificação, precisa retornar True

len(X_test[X_test.species == 'Gentoo']) == 36

True

### Hot Encode



In [ ]:
# Identify non-numeric columns in X_train
non_numeric_cols = X_train.select_dtypes(exclude=np.number).columns

# Apply one-hot encode to non-numeric columns
encoder = OneHotEncoder(sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train[non_numeric_cols])
X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(non_numeric_cols))

# Drop original non-numeric columns and concatenate encoded columns
X_train = X_train.drop(columns=non_numeric_cols).reset_index(drop=True)
X_train = pd.concat([X_train, X_train_encoded], axis=1)

X_test_encoded = encoder.transform(X_test[non_numeric_cols])
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(non_numeric_cols))

# Drop original non-numeric columns and concatenate encoded columns
X_test = X_test.drop(columns=non_numeric_cols).reset_index(drop=True)
X_test = pd.concat([X_test, X_test_encoded], axis=1)


In [ ]:
print(f'Número de colunas de X_train: {len(X_train.columns)}')

Número de colunas de X_train: 9


In [ ]:
#@markdown Verificação, precisa retornar True

X_train.species_Adelie.sum() == 102

np.True_

### Normalização




In [ ]:
# Normalizando os dados
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
print(f'Soma dos valores de np.round(X_train[1].sum(),0): {np.round(X_train[1].sum(),0)}')

Soma dos valores de np.round(X_train[1].sum(),0): 3.0


In [ ]:
#@markdown Verificação, precisa retornar True
np.round(X_test[1].sum(), 2) == -1.2

np.True_

## Modelo MLP 1

Implemente um algoritmo MLP para classificação da ilha (`island`) de origem dos pinguins. Empregue `random_state=1`, uma rede com camadas ocultas de 8, 16 e 8 neurônios respectivamente e número máximo de iterações 5000. Não empregue outros parâmetros além dos solicitados.

Obtenha ao final a acuracidade do modelo, o `classification report` e a matriz de confusão.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Definição
clf = MLPClassifier(random_state=1, hidden_layer_sizes=(8,16,8), max_iter=5000)

# Treinamento
clf.fit(X_train, y_train)

# Avaliação
y_pred = clf.predict(X_test)

print('\nClassification Report:\n')
print(classification_report(y_test, y_pred))

print('\nConfusion Matrix:\n')
print(confusion_matrix(y_test, y_pred))



Classification Report:

              precision    recall  f1-score   support

      Biscoe       0.79      0.78      0.78        49
       Dream       0.71      0.68      0.69        37
   Torgersen       0.41      0.50      0.45        14

    accuracy                           0.70       100
   macro avg       0.64      0.65      0.64       100
weighted avg       0.71      0.70      0.70       100


Confusion Matrix:

[[38  6  5]
 [ 7 25  5]
 [ 3  4  7]]


In [ ]:
#@markdown Verificação, precisa retornar True
np.round(clf.coefs_[0].sum(),2) == -3.77

np.True_

## Modelo MLP 2

Altere o modelo anterior para uma rede com 1000 e 500 elementos de camadas internas, a função de ativação `relu`. Matenha os demais parâmetros e empregue o mesmo conjunto de Treinamento e Teste do modelo anterior.

In [ ]:
# Definição
clf2 = MLPClassifier(random_state=1, hidden_layer_sizes=(1000,500), max_iter=5000, activation='relu')

# Treinamento
clf2.fit(X_train, y_train)

# Avaliação
y_pred = clf2.predict(X_test)

print('\nClassification Report:\n')
print(classification_report(y_test, y_pred))

print('\nConfusion Matrix:\n')
print(confusion_matrix(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

      Biscoe       0.76      0.80      0.78        49
       Dream       0.74      0.68      0.70        37
   Torgersen       0.27      0.29      0.28        14

    accuracy                           0.68       100
   macro avg       0.59      0.59      0.59       100
weighted avg       0.68      0.68      0.68       100


Confusion Matrix:

[[39  4  6]
 [ 7 25  5]
 [ 5  5  4]]


In [ ]:
#@markdown Verificação, precisa retornar True
np.round(clf.coefs_[1].sum(),2) == 3.85

np.True_

## Predição

Faça a predição dos seguinte pinguim:

```
species                 Adelie
bill_length_mm            40.0
bill_depth_mm             18.0
flipper_length_mm        185.0
body_mass_g             3900.0
sex                       Male
```

Qual a probabilidade de cada uma das classes na predição?

In [ ]:
X_new = pd.DataFrame()

X_new['species']=['Adelie']

X_new['bill_length_mm']=40.0
X_new['bill_depth_mm']=18.0
X_new['flipper_length_mm']=185.0
X_new['body_mass_g']=3900.0
X_new['sex']='Male'

X_new.head()

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,40.0,18.0,185.0,3900.0,Male


In [ ]:
X_encoded = encoder.transform(X_new[non_numeric_cols])
X_encoded = pd.DataFrame(X_encoded, columns=encoder.get_feature_names_out())
X_new = pd.concat([X_new.drop(columns=non_numeric_cols), X_encoded], axis=1)
display(X_new.head())

X_new = scaler.transform(X_new)

y_pred = clf.predict(X_new)

print(y_pred)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,species_Adelie,species_Chinstrap,species_Gentoo,sex_Female,sex_Male
0,40.0,18.0,185.0,3900.0,1.0,0.0,0.0,0.0,1.0


['Biscoe']


In [ ]:
clf.classes_

array(['Biscoe', 'Dream', 'Torgersen'], dtype='<U9')

In [ ]:
clf.predict_proba(X_new)

array([[0.42567587, 0.35276353, 0.2215606 ]])

In [ ]:
#@markdown Verificação, precisa retornar True
np.round(clf.predict_proba(X_new)[0][2],2) == 0.22

np.True_